In [ ]:
# =============================================================
# CELL 1 — Setup, configuration & dependency check (v3)
# =============================================================
# Installs pinned deps, imports, defines ONE central CFG dict that
# every later cell reads from, and runs a silent traja environment
# guard (raises clearly if a version mismatch breaks the API).
# -------------------------------------------------------------

!pip install -q "traja==25.0.1" "shapely==2.1.2"

import sys, io, os, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import ipywidgets as widgets
from IPython.display import display
import traja
from shapely.geometry import Polygon, Point   # Point: v2's analyze_roi_time omitted it

# ----------------------------------------------------------------
# CENTRAL CONFIG — edit everything here, not scattered through cells
# ----------------------------------------------------------------
CFG = {
    # data
    "DATA_ROOT":         Path("/content/data"),
    # spatial normalisation
    "CANONICAL_SIDE":    "Right",   # flip so the social side always ends here
    "ZONE_METHOD":       "width",   # "width" (equal-width thirds) or "area"
    # segments (frames) — global defaults; editable in the config UI cell
    "SEG_BEFORE":        (0, 3500),
    "SEG_AFTER":         (5500, 9000),
    # preprocessing
    "DOWNSAMPLE":        False,
    "TARGET_FPS":        10.0,
    "DOWNSAMPLE_METHOD": "downsample",   # "downsample" or "resample"
    "SMOOTH":            True,
    "SMOOTH_POLY":       3,
    # visualisation
    "HEATMAP_BINS":      40,
    "LUT_PERCENTILE":    99,        # locked-scale clip; lower (e.g. 95) lifts midtones
    "CMAP":              "hot",
    # zone semantics after flip: code 3 = social (right), 2 = centre, 1 = far
    "ZONE_NAME":         {3: "social", 2: "centre", 1: "far"},
}
CFG["DATA_ROOT"].mkdir(parents=True, exist_ok=True)

print("Python     :", sys.version.split()[0])
print("numpy      :", np.__version__, "| pandas:", pd.__version__,
      "| matplotlib:", matplotlib.__version__)
print("traja      :", traja.__version__, "| (pinned 25.0.1)")
print("-" * 56)

# --- Silent environment guard: confirms the four traja calls work ---
_demo = traja.TrajaDataFrame({"time": np.arange(50) / 30.0,
                              "x": np.cos(np.linspace(0, 6, 50)),
                              "y": np.sin(np.linspace(0, 6, 50))})
_demo.fps = 30
try:
    traja.trajectory.distance(_demo)
    traja.trajectory.get_derivatives(_demo)
    traja.trajectory.smooth_sg(_demo, w=11, p=3)
    _demo.traja.trip_grid(bins=10)
    plt.close("all")          # silence trip_grid's side-effect figure
    print("✓ traja environment OK — CFG ready, proceed to Cell 2.")
except Exception as e:
    plt.close("all")
    raise RuntimeError(f"traja API check failed ({e}). "
                       "Likely a version mismatch — re-run the install cell.") from e

In [ ]:
# =============================================================
# CELL 2 — Data-access helpers (scan / probe / resolve / parse)
# =============================================================
# Library cell: DEFINES discovery helpers only. Scanning is triggered
# by Cell 3. Now parses nickname + genotype + social side from the
# session FOLDER name (pattern: date_genotype_fish_side, e.g.
# '20250829_dNLS_F1_L'), so social_side and flip are set automatically
# — no manual Left/Right entry needed. Requires CFG, np, pd, Path (Cell 1).
# -------------------------------------------------------------
DATA_ROOT = CFG["DATA_ROOT"]
_SIDE_MAP = {"L": "Left", "R": "Right"}

def resolve_asset(npy_path, filename):
    """Find background.png / ROI_mask.png near a trajectories.npy."""
    npy_path = Path(npy_path)
    for c in (npy_path.parent.parent / "preprocessing" / filename,
              npy_path.parent / "preprocessing" / filename,
              npy_path.parent / filename,
              npy_path.parent.parent / filename):
        if c.exists():
            return str(c)
    return ""

def parse_session_name(name, canonical_side=None):
    """Parse 'date_genotype_fish_side' -> (genotype, social_side, flip).
    e.g. '20250829_dNLS_F1_L' -> ('dNLS', 'Left', True if canonical='Right').
    Unknown tokens -> None."""
    canonical_side = canonical_side or CFG.get("CANONICAL_SIDE", "Right")
    parts = str(name).split("_")
    social_side = _SIDE_MAP.get(parts[-1].upper()) if parts else None
    genotype = parts[1] if len(parts) >= 2 else None
    flip = (social_side is not None) and (social_side != canonical_side)
    return genotype, social_side, flip

def probe_npy(npy_path):
    """Light metadata probe — does NOT build a traja df."""
    info = {"meta_name": Path(npy_path).stem, "fps": None, "n_frames": None,
            "n_individuals": None, "body_length": None, "width": None,
            "height": None, "error": ""}
    try:
        data = np.load(npy_path, allow_pickle=True)
        if data.shape == ():
            c = data.item()
            traj = c.get("trajectories")
            if traj is not None:
                info["n_frames"] = traj.shape[0]
                info["n_individuals"] = traj.shape[1] if traj.ndim >= 3 else 1
            info["fps"] = c.get("frames_per_second")
            info["body_length"] = c.get("body_length")
            info["width"] = c.get("width")
            info["height"] = c.get("height")
            vps = c.get("video_paths")
            if vps:
                info["meta_name"] = Path(str(vps[0]).replace("\\", "/")).stem
        else:
            info["n_frames"] = data.shape[0]
            info["n_individuals"] = data.shape[1] if data.ndim >= 3 else 1
    except Exception as e:
        info["error"] = str(e)
    return info

def scan_sessions(root=None):
    """Walk root for trajectories.npy -> registry DataFrame.
    group     = top-level folder under root (fallback: parsed genotype)
    nickname  = session folder name (e.g. 20250829_dNLS_F1_L)
    social_side / flip = parsed from that folder name."""
    root = Path(root or DATA_ROOT)
    rows, no_side = [], []
    for npy_path in sorted(root.rglob("trajectories.npy")):
        rel = npy_path.relative_to(root)
        # session folder holds trajectories/ + preprocessing/
        sess_dir = npy_path.parent
        if sess_dir.name.lower() == "trajectories":
            sess_dir = sess_dir.parent
        session_name = sess_dir.name
        group = rel.parts[0] if len(rel.parts) > 1 else "(ungrouped)"
        genotype, social_side, flip = parse_session_name(session_name)
        if group == "(ungrouped)" and genotype:
            group = genotype
        if social_side is None:
            no_side.append(session_name)
        m = probe_npy(npy_path)
        rows.append({
            "group": group, "nickname": session_name, "meta_name": m["meta_name"],
            "social_side": social_side, "flip": flip, "npy": str(npy_path),
            "background": resolve_asset(npy_path, "background.png"),
            "roi_mask":   resolve_asset(npy_path, "ROI_mask.png"),
            "fps": m["fps"], "n_frames": m["n_frames"], "n_ind": m["n_individuals"],
            "body_len_px": m["body_length"], "w": m["width"], "h": m["height"],
            "error": m["error"],
        })
    df = pd.DataFrame(rows)
    if no_side:
        print(f"⚠ {len(no_side)} session(s) with no L/R suffix parsed: "
              + ", ".join(no_side[:5]) + (" …" if len(no_side) > 5 else ""))
    return df

registry = pd.DataFrame()
print("Data-access helpers ready (with filename parsing). Upload & scan in Cell 3.")

In [ ]:
# =============================================================
# CELL 3 — Data source: Google Drive (point to parent folder)
# =============================================================
# Mounts Drive, then lets you pick the folder that CONTAINS your group
# subfolders (e.g. a folder holding dNLS/ and wtTDP/). Scanning reads
# the group from the first subfolder level, so:
#     <picked folder>/dNLS/<session>/...   -> group "dNLS"
#     <picked folder>/wtTDP/<session>/...  -> group "wtTDP"
# The dropdown only lists folders that actually contain trajectories.npy
# somewhere inside, so you can't pick an empty one.
# Requires: CFG (Cell 1), scan_sessions (Cell 2).
# -------------------------------------------------------------
from google.colab import drive
from IPython.display import clear_output

drive.mount("/content/drive", force_remount=True)          # idempotent; re-mount is fine

# Where to look. My Drive by default; change to Shareddrives if needed.
SEARCH_BASE = Path("/content/drive/MyDrive")
# SEARCH_BASE = Path("/content/drive/Shareddrives")   # for a Shared drive

# Candidate PARENT folders = any dir that has >=2 subfolders each holding
# a trajectories.npy (i.e. looks like a "contains groups" folder), plus
# any dir containing trajectories.npy at all (fallback).
def _candidate_roots(base, limit=400):
    cands = set()
    for npy in list(base.rglob("trajectories.npy"))[:limit]:
        # session folder is npy.parent.parent (session/trajectories/trajectories.npy)
        # group folder is its parent; the "root" is the group's parent.
        grp = npy.parent.parent.parent          # .../<root>/<group>/<session>
        root = grp.parent
        cands.add(str(grp)); cands.add(str(root))
    return sorted(cands)

choices = _candidate_roots(SEARCH_BASE)
if not choices:
    raise FileNotFoundError(
        f"No trajectories.npy found under {SEARCH_BASE}. "
        "Is the upload finished? Check the path / Shared-drive setting.")

picker = widgets.Dropdown(options=choices, description="Data root:",
                          layout=widgets.Layout(width="95%"))
scan_btn = widgets.Button(description="Scan this folder", icon="cogs",
                          button_style="success")
ui = widgets.VBox([widgets.HTML("<b>Pick the folder that contains your "
                                "group subfolders (dNLS, wtTDP):</b>"),
                   picker, scan_btn])

def _do_scan(_):
    clear_output(wait=True); display(ui)
    global registry
    CFG["DRIVE_ROOT"] = Path(picker.value)
    registry = scan_sessions(CFG["DRIVE_ROOT"])
    print(f"Scanned {len(registry)} session(s) from {picker.value}\n")
    if len(registry):
        counts = registry["group"].value_counts()
        print("Sessions per group (this is your grouping — check it):")
        for g, n in counts.items():
            flag = "  ⚠ single session" if n == 1 else ""
            print(f"   {g}: {n}{flag}")
        display(registry)
    else:
        print("No sessions found here — try the folder one level up.")

scan_btn.on_click(_do_scan)
display(ui)

In [ ]:
# =============================================================
# CELL 4 — Geometry, zones & occupancy-stats library
# =============================================================
# Mask loading, zone derivation (thirds), point classification, and
# occupancy stats. Replaces v2's ray-casting with direct mask lookup.
# Reads CFG["ZONE_METHOD"] / CFG["ZONE_NAME"]. Requires Cell 1.
# -------------------------------------------------------------
def load_roi_mask(path, flip=False, thresh=0.5):
    """White arena on black -> boolean (H, W). Handles RGB/RGBA/gray."""
    img = mpimg.imread(str(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    mask = img > thresh
    if flip:
        mask = mask[:, ::-1]
    return mask

def load_background(path, flip=False):
    """Background -> grayscale (H, W) for plotting underlays."""
    if not path:
        return None
    img = mpimg.imread(str(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    if flip:
        img = img[:, ::-1]
    return img

def arena_bounds(mask):
    cols = np.where(mask.any(axis=0))[0]
    rows = np.where(mask.any(axis=1))[0]
    return int(cols.min()), int(cols.max()), int(rows.min()), int(rows.max())

def build_zone_map(mask, method=None):
    method = method or CFG["ZONE_METHOD"]
    H, W = mask.shape
    bounds = arena_bounds(mask)          # compute once
    min_x, max_x, _, _ = bounds
    X = np.broadcast_to(np.arange(W), (H, W))
    if method == "width":
        span = (max_x - min_x + 1) / 3.0
        b1, b2 = min_x + span, min_x + 2 * span
    elif method == "area":
        col_counts = mask.sum(axis=0).astype(float)
        csum, total = np.cumsum(col_counts), col_counts.sum()
        b1 = int(np.searchsorted(csum, total / 3.0))
        b2 = int(np.searchsorted(csum, 2 * total / 3.0))
    else:
        raise ValueError("ZONE_METHOD must be 'width' or 'area'")
    zone = np.zeros((H, W), dtype=np.uint8)
    zone[mask & (X < b1)] = 1
    zone[mask & (X >= b1) & (X < b2)] = 2
    zone[mask & (X >= b2)] = 3
    return zone, (b1, b2), bounds

def classify_points(x, y, zone_map):
    """Trajectory points -> zone codes (0..3). NaNs -> 0. Vectorised."""
    H, W = zone_map.shape
    x, y = np.asarray(x, float), np.asarray(y, float)
    codes = np.zeros(len(x), dtype=np.uint8)
    valid = ~(np.isnan(x) | np.isnan(y))
    xi = np.clip(np.round(x[valid]).astype(int), 0, W - 1)
    yi = np.clip(np.round(y[valid]).astype(int), 0, H - 1)
    codes[valid] = zone_map[yi, xi]
    return codes, valid

def zone_stats(x, y, zone_map, fps):
    """Frames/time/percent per zone. % uses VALID frames as denominator."""
    codes, valid = classify_points(x, y, zone_map)
    n_valid = int(valid.sum())
    keymap = {1: "left", 2: "middle", 3: "right"}   # internal keys
    out = {"total_frames": len(x), "valid_frames": n_valid}
    for code, k in keymap.items():
        v = int((codes == code).sum())
        out[f"frames_{k}"] = v
        out[f"time_{k}"]   = v / fps
        out[f"pct_{k}"]    = 100.0 * v / n_valid if n_valid else 0.0
    roi = int((codes > 0).sum())
    out["frames_roi"] = roi
    out["pct_roi"] = 100.0 * roi / n_valid if n_valid else 0.0
    return out

def _seg_slice(trj, start, end):
    """Frame-range slice, clamped, end-exclusive."""
    n = len(trj)
    s = max(0, min(int(start), n))
    e = max(s, min(int(end), n))
    return trj.iloc[s:e]

print("Geometry/zones/stats library ready.")

In [ ]:
# =============================================================
# CELL 5 — QC contact sheet (compact, half-coloured trajectory)
# =============================================================
# Grid of session thumbnails: background + zone tint + trajectory
# coloured by video HALF (1st half = blue, 2nd half = red). Honours
# flip / social_side / exclude columns (auto-set in Cell 2), so this
# also verifies the auto-flip. Requires Cell 4 + registry.
# -------------------------------------------------------------
from matplotlib.lines import Line2D

ZONE_LABELS  = {1: "non-social", 2: "centre", 3: "SOCIAL"}
HALF_COLOURS = {"first": "#FFFF00", "second": "#FF00FF"}   # yellow=early, magenta=late

def _raw_xy(npy_path, individual_idx=0):
    data = np.load(npy_path, allow_pickle=True)
    traj = data.item().get("trajectories") if data.shape == () else data
    if traj is None:
        return None, None
    if traj.ndim == 3:
        idx = individual_idx if individual_idx < traj.shape[1] else 0
        traj = traj[:, idx, :]
    return traj[:, 0], traj[:, 1]

def contact_sheet(reg, ncols=4, panel=2.6, show_traj=True, max_points=1200):
    if reg is None or len(reg) == 0:
        print("Registry empty — upload/scan in Cell 3 first."); return
    n = len(reg); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel*ncols, panel*nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, reg.iterrows()):
        flip = bool(row.get("flip", False))
        excluded = bool(row.get("exclude", False))
        try:
            mask = load_roi_mask(row["roi_mask"], flip=flip)
            bg = load_background(row["background"], flip=flip)
            zone, (b1, b2), _ = build_zone_map(mask)
            H, W = mask.shape
            if bg is not None:
                ax.imshow(bg, cmap="gray", origin="upper")
            ax.imshow(np.ma.masked_where(zone == 0, zone), cmap="brg",
                      alpha=0.25, origin="upper", vmin=1, vmax=3)
            ax.axvline(b1, color="cyan", ls="--", lw=0.8)
            ax.axvline(b2, color="cyan", ls="--", lw=0.8)
            if show_traj:
                x, y = _raw_xy(row["npy"])
                if x is not None:
                    x = np.asarray(x, float); y = np.asarray(y, float)
                    if flip: x = (W - 1) - x
                    half = len(x) // 2
                    step = max(1, len(x) // max_points)
                    ax.plot(x[:half][::step], y[:half][::step],
                            color=HALF_COLOURS["first"],  lw=0.4, alpha=0.7)
                    ax.plot(x[half:][::step], y[half:][::step],
                            color=HALF_COLOURS["second"], lw=0.4, alpha=0.7)
            ax.set_xlim(0, W); ax.set_ylim(H, 0)
            title = f"{row['group']} / {row['nickname']}"
            if pd.notna(row.get("social_side")):
                title += f"\nsocial={row['social_side']}{' [flip]' if flip else ''}"
            ax.set_title(title, fontsize=6.5, color="red" if excluded else "black")
            if excluded:
                ax.text(W/2, H/2, "EXCLUDED", ha="center", va="center",
                        color="red", fontsize=9, fontweight="bold")
        except Exception as e:
            ax.text(0.5, 0.5, f"{row['nickname']}\n{type(e).__name__}",
                    ha="center", va="center", fontsize=6, color="red",
                    transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis("off")
    if show_traj:
        handles = [Line2D([0],[0], color=HALF_COLOURS["first"],  lw=2, label="1st half"),
                   Line2D([0],[0], color=HALF_COLOURS["second"], lw=2, label="2nd half")]
        fig.legend(handles=handles, loc="upper right", ncol=2, fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()

contact_sheet(registry)

In [ ]:
# =============================================================
# CELL 6 — Config: global segments + confirm/override side + exclude
# =============================================================
# Social side is AUTO-PARSED in Cell 2 from the _L/_R filename suffix,
# so here the dropdowns PRE-FILL from that parsed value (override only
# if a file was mis-named). Real manual inputs = global segment frames
# + exclude. Apply writes seg/social_side/flip/exclude into `registry`;
# flip is recomputed from the (possibly overridden) side.
# Requires: CFG (Cell 1) + registry with social_side (Cell 2/3).
# -------------------------------------------------------------
from IPython.display import clear_output

CANONICAL_SIDE = CFG["CANONICAL_SIDE"]
_fps = float(registry["fps"].dropna().iloc[0]) if registry["fps"].notna().any() else 30.0
_nframes = int(registry["n_frames"].dropna().iloc[0]) if registry["n_frames"].notna().any() else 9000

def _int_field(label, val):
    return widgets.BoundedIntText(value=val, min=0, max=_nframes, step=1,
                                  description=label, layout=widgets.Layout(width="46%"))

before_start = _int_field("Before start", CFG["SEG_BEFORE"][0])
before_end   = _int_field("Before end",   CFG["SEG_BEFORE"][1])
after_start  = _int_field("After start",  CFG["SEG_AFTER"][0])
after_end    = _int_field("After end",    CFG["SEG_AFTER"][1])
secs_lbl = widgets.HTML()

def _refresh_secs(*_):
    f = _fps
    secs_lbl.value = (f"<i>≈ at {f:.3f} fps  Before: {before_start.value/f:.1f}–"
                      f"{before_end.value/f:.1f}s | After: {after_start.value/f:.1f}–"
                      f"{after_end.value/f:.1f}s | gap: {(after_start.value-before_end.value)/f:.1f}s</i>")
for w in (before_start, before_end, after_start, after_end):
    w.observe(_refresh_secs, "value")
_refresh_secs()

side_widgets, excl_widgets, rows = {}, {}, [
    widgets.HTML("<b>Global segments (frames)</b> — same for every video"),
    widgets.HBox([before_start, before_end]),
    widgets.HBox([after_start, after_end]), secs_lbl, widgets.HTML("<hr>"),
    widgets.HTML("<b>Per-video:</b> side is auto-parsed (override only if mis-named) &amp; exclude")]

for _, r in registry.iterrows():
    nick = r["nickname"]
    parsed = r.get("social_side")
    # pre-fill from parsed side; fall back to canonical if it didn't parse
    default_side = parsed if parsed in ("Left", "Right") else CANONICAL_SIDE
    flag = "" if parsed in ("Left", "Right") else "  ⚠ no _L/_R parsed"
    side = widgets.Dropdown(options=["Left", "Right"], value=default_side,
                            layout=widgets.Layout(width="22%"))
    excl = widgets.Checkbox(value=bool(r.get("exclude", False)), indent=False,
                            layout=widgets.Layout(width="16%"))
    side_widgets[nick], excl_widgets[nick] = side, excl
    rows.append(widgets.HBox([
        widgets.HTML(f"{r['group']} / {nick}{flag}", layout=widgets.Layout(width="55%")),
        side, excl]))

apply_btn = widgets.Button(description="Apply config", icon="check", button_style="success")

def _apply(_):
    clear_output(wait=True); display(ui)
    if not (before_start.value < before_end.value <= after_start.value < after_end.value):
        print("⚠ Need before_start < before_end ≤ after_start < after_end."); return
    registry["seg_before_start"] = before_start.value
    registry["seg_before_end"]   = before_end.value
    registry["seg_after_start"]  = after_start.value
    registry["seg_after_end"]    = after_end.value
    registry["social_side"] = [side_widgets[n].value for n in registry["nickname"]]
    registry["exclude"]     = [excl_widgets[n].value for n in registry["nickname"]]
    registry["flip"] = registry["social_side"] != CANONICAL_SIDE   # recompute from side
    inc = (~registry["exclude"]).sum()
    print(f"Config applied. Canonical = {CANONICAL_SIDE}. Included {inc}/{len(registry)} "
          f"(flipping {registry['flip'][~registry['exclude']].sum()}).")
    display(registry[["group","nickname","social_side","flip","exclude",
                      "seg_before_start","seg_before_end","seg_after_start","seg_after_end"]])

apply_btn.on_click(_apply)
ui = widgets.VBox(rows + [apply_btn])
display(ui)

In [ ]:
# =============================================================
# CELL 7 — Load & preprocess sessions into traja DataFrames
# =============================================================
# Slices focal fish, builds 'time' column, applies flip, optional
# downsample/smooth (from CFG). Caches in `loaded`. Skips excluded.
# Requires Cells 1,2,4 + applied config (Cell 6).
# -------------------------------------------------------------
def _extract_focal_xy(npy_path, individual_idx=0):
    data = np.load(npy_path, allow_pickle=True)
    meta = {}
    if data.shape == ():
        c = data.item(); meta = c; traj = None
        for k in ("trajectories","trajectories_smooth","trajectories_filtered",
                  "tracks","positions","data"):
            if c.get(k) is not None:
                traj = c[k]; break
    else:
        traj = data
    if traj is None:
        raise ValueError(f"No trajectory data in {npy_path}")
    if traj.ndim == 3:
        idx = individual_idx if individual_idx < traj.shape[1] else 0
        traj = traj[:, idx, :]
    return traj[:, 0].astype(float), traj[:, 1].astype(float), meta

def _smooth(trj):
    w = min(101, (len(trj)//2)*2 + 1)
    if w < CFG["SMOOTH_POLY"] + 2:
        return trj
    return traja.trajectory.smooth_sg(trj, w=w, p=CFG["SMOOTH_POLY"])

def build_trajectory(row):
    fps = float(row["fps"]) if pd.notna(row["fps"]) else 30.0
    x, y, meta = _extract_focal_xy(row["npy"])
    if bool(row.get("flip", False)):
        W = load_roi_mask(row["roi_mask"]).shape[1]
        x = (W - 1) - x
    df = pd.DataFrame({"time": np.arange(len(x))/fps, "x": x, "y": y})
    trj = traja.TrajaDataFrame(df); trj.fps = fps
    processing = "raw"
    if CFG["DOWNSAMPLE"] and 0 < CFG["TARGET_FPS"] < fps:
        tf = CFG["TARGET_FPS"]
        if CFG["DOWNSAMPLE_METHOD"] == "downsample":
            factor = max(1, int(round(fps/tf)))
            d = trj.iloc[::factor].copy(); d["time"] = np.arange(len(d))/tf
            trj = traja.TrajaDataFrame(d); trj.fps = tf; processing = f"downsampled x{factor}"
        else:
            d = trj.copy(); d["time"] = pd.to_timedelta(d["time"], unit="s")
            d = d.set_index("time").resample(f"{1000/tf:.0f}ms").mean().reset_index().dropna()
            d["time"] = d["time"].dt.total_seconds()
            trj = traja.TrajaDataFrame(d); trj.fps = tf; processing = "time-averaged"
    if CFG["SMOOTH"]:
        keep = trj.fps; trj = _smooth(trj); trj.fps = keep; processing += "+smoothed"
    return trj, {"fps": getattr(trj,"fps",fps), "body_length_px": meta.get("body_length"),
                 "n_frames": len(trj), "processing": processing,
                 "pct_tracked": 100.0*np.mean(~(np.isnan(x)|np.isnan(y)))}

loaded = {}
work = registry[~registry.get("exclude", False)] if "exclude" in registry else registry
for _, row in work.iterrows():
    nick = row["nickname"]
    try:
        trj, info = build_trajectory(row)
        loaded[nick] = {"trj": trj, "info": info, "row": row}
        bl = info["body_length_px"]
        print(f"✓ {nick:24s} {info['n_frames']:5d} fr @ {info['fps']:.3f} | "
              f"BL={bl:.1f}px | {info['pct_tracked']:.0f}% | {info['processing']}"
              if bl else f"✓ {nick:24s} BL=missing")
    except Exception as e:
        print(f"✗ {nick:24s} FAILED: {type(e).__name__}: {e}")
print(f"\nLoaded {len(loaded)} session(s).")

In [ ]:
# =============================================================
# CELL 8 — Occupancy stats: per-session + per-group (+ Excel export)
# =============================================================
# "before"/"after" = PHASE; "social" = stimulus-side third (right after
# flip). Builds tidy tables (no print-loop) + adds per-phase tracking%
# so exact-100% fish can be sanity-checked. Download button writes an
# Excel workbook for you to plot from. % uses VALID frames as denominator.
# Requires Cells 1,4,7.
# -------------------------------------------------------------
import datetime as _dt

ZONE_NAME = CFG["ZONE_NAME"]        # {3:'social',2:'centre',1:'far'}
_keymap = {3: "right", 2: "middle", 1: "left"}

def session_occupancy(nick):
    e = loaded[nick]; trj, row, info = e["trj"], e["row"], e["info"]; fps = info["fps"]
    zmap, _, _ = build_zone_map(load_roi_mask(row["roi_mask"], flip=bool(row.get("flip", False))))
    phases = {"before": (row["seg_before_start"], row["seg_before_end"]),
              "after":  (row["seg_after_start"],  row["seg_after_end"])}
    occ, trk_by_phase, long_rows = {}, {}, []
    for phase, (s, en) in phases.items():
        sub = _seg_slice(trj, s, en)
        st = zone_stats(sub["x"].values, sub["y"].values, zmap, fps)
        trk = 100.0*np.mean(~(np.isnan(sub["x"].values)|np.isnan(sub["y"].values))) if len(sub) else np.nan
        trk_by_phase[phase] = trk
        for code, zname in ZONE_NAME.items():
            occ[(phase, zname)] = st[f"pct_{_keymap[code]}"]
            long_rows.append({"group": row["group"], "nickname": nick, "phase": phase,
                              "zone": zname, "occupancy_pct": st[f"pct_{_keymap[code]}"],
                              "time_s": st[f"time_{_keymap[code]}"],
                              "valid_frames": st["valid_frames"], "tracking_pct": trk})
    b, a = occ[("before","social")], occ[("after","social")]
    ratio = (a/b) if b > 0 else (np.inf if a > 0 else np.nan)
    return pd.DataFrame(long_rows), {
        "group": row["group"], "nickname": nick,
        "occupancy_before": b, "occupancy_after": a, "ratio": ratio, "diff": a - b,
        "track_before_pct": trk_by_phase["before"], "track_after_pct": trk_by_phase["after"]}

# ---- Build tables ----
_long, _summ = [], []
for nick in loaded:
    l, s = session_occupancy(nick); _long.append(l); _summ.append(s)
occupancy_long = pd.concat(_long, ignore_index=True)
social_summary = pd.DataFrame(_summ)

def _sem(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return x.std(ddof=1)/np.sqrt(len(x)) if len(x) > 1 else 0.0

grp_rows = []
for grp, g in social_summary.groupby("group"):
    fr = g["ratio"].replace([np.inf,-np.inf], np.nan).dropna()
    mb, ma = g["occupancy_before"].mean(), g["occupancy_after"].mean()
    grp_rows.append({"group": grp, "n": len(g),
        "before_mean": mb, "before_sem": _sem(g["occupancy_before"]),
        "after_mean": ma, "after_sem": _sem(g["occupancy_after"]),
        "diff_mean": g["diff"].mean(), "diff_sem": _sem(g["diff"]),
        "ratio_mean_of_fish": fr.mean() if len(fr) else np.nan,
        "ratio_of_means": (ma/mb) if mb > 0 else np.nan})
group_summary = pd.DataFrame(grp_rows)

# ---- Display: per-session table (rounded, with tracking guard) ----
_disp = social_summary.copy()
for c in ("occupancy_before","occupancy_after","diff","track_before_pct","track_after_pct"):
    _disp[c] = _disp[c].round(1)
_disp["ratio"] = _disp["ratio"].round(2)
_disp["⚠"] = np.where(_disp[["track_before_pct","track_after_pct"]].min(axis=1) < 90,
                      "low tracking", "")
print("Per-session social-third occupancy (% of valid frames):")
display(_disp)

# ---- Display: per-group summary as mean ± SEM ----
_pm = lambda m, s: f"{m:.1f} ± {s:.1f}"
group_display = pd.DataFrame({
    "group": group_summary["group"], "n": group_summary["n"],
    "before %": [_pm(m, s) for m, s in zip(group_summary.before_mean, group_summary.before_sem)],
    "after %":  [_pm(m, s) for m, s in zip(group_summary.after_mean,  group_summary.after_sem)],
    "Δ (pts)":  [_pm(m, s) for m, s in zip(group_summary.diff_mean,   group_summary.diff_sem)],
    "ratio (mean of fish)": group_summary.ratio_mean_of_fish.round(2),
    "ratio (of means)":     group_summary.ratio_of_means.round(2)})
print("\nPer-group summary (mean ± SEM):")
display(group_display)

# ---- Export: Excel workbook + download button ----
def export_stats(path="social_preference_stats.xlsx"):
    # config/provenance sheet so the spreadsheet is self-documenting
    cfg_items = {k: (str(v) if isinstance(v, (tuple, list, dict, Path)) else v)
                 for k, v in CFG.items()}
    cfg_items["exported_at"] = _dt.datetime.now().isoformat(timespec="seconds")
    cfg_items["n_sessions"] = len(social_summary)
    config_df = pd.DataFrame(sorted(cfg_items.items()), columns=["setting", "value"])
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as xl:
            social_summary.to_excel(xl, sheet_name="per_session", index=False)
            group_summary.to_excel(xl,  sheet_name="per_group",  index=False)
            occupancy_long.to_excel(xl, sheet_name="occupancy_long", index=False)
            config_df.to_excel(xl,      sheet_name="config", index=False)
        print(f"Wrote {path}  (4 sheets)")
        try:
            from google.colab import files
            files.download(path)
        except Exception:
            print("Not in Colab — file saved to the working directory.")
    except Exception as e:
        # Fallback: CSVs if Excel engine unavailable
        social_summary.to_csv("per_session.csv", index=False)
        group_summary.to_csv("per_group.csv", index=False)
        occupancy_long.to_csv("occupancy_long.csv", index=False)
        print(f"Excel failed ({e}); wrote per_session.csv / per_group.csv / occupancy_long.csv")

_dl_btn = widgets.Button(description="Download stats (.xlsx)", icon="download",
                         button_style="success")
_dl_btn.on_click(lambda _: export_stats())
display(_dl_btn)

In [ ]:
# =============================================================
# CELL 8b — Occupancy: combined paired plot (both groups, one axes)
# =============================================================
# One panel. Per group: before+after columns close together, a gap,
# then the next group. Lines connect each fish before->after, coloured
# by GROUP only. Group mean ± 95% CI overlaid directly on each column.
# Δ (mean change) with 95% CI annotated above each cluster. Estimation
# plot, NOT a test. Low-tracking fish drawn hollow (QC, same colour).
# Requires: social_summary (Cell 8), scipy.
# -------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

GROUP_PALETTE = {"dNLS": "#1f77b4", "wtTDP": "#d62728"}
_FALLBACK = ["#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]

def _mean_ci(x, conf=0.95):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    n = len(x)
    if n == 0: return np.nan, np.nan, np.nan
    m = float(x.mean())
    if n < 2:  return m, np.nan, np.nan
    se = x.std(ddof=1) / np.sqrt(n)
    h = se * stats.t.ppf((1 + conf) / 2, n - 1)
    return m, m - h, m + h

def plot_occupancy_combined(summary=None, conf=0.95, low_track=90.0,
                            cluster_gap=2.0, pair_gap=0.55, show_mean_bar=False):
    summary = summary if summary is not None else social_summary
    groups = list(dict.fromkeys(summary["group"]))
    for i, g in enumerate(groups):
        GROUP_PALETTE.setdefault(g, _FALLBACK[i % len(_FALLBACK)])
    rng = np.random.default_rng(0)

    fig, ax = plt.subplots(figsize=(3.0 * len(groups) + 2.5, 6))
    xticks, xlabels = [], []

    for gi, g in enumerate(groups):
        col = GROUP_PALETTE[g]; gdf = summary[summary["group"] == g]
        xb = gi * cluster_gap; xa = xb + pair_gap

        for _, r in gdf.iterrows():
            b, a = r["occupancy_before"], r["occupancy_after"]
            jb, ja = rng.uniform(-0.05, 0.05, 2)
            low = min(r.get("track_before_pct", 100), r.get("track_after_pct", 100)) < low_track
            ax.plot([xb + jb, xa + ja], [b, a], color=col, alpha=0.28, lw=1.1, zorder=1)
            ax.scatter([xb + jb, xa + ja], [b, a], s=26,
                       facecolors="white" if low else col, edgecolors=col,
                       linewidths=1.2, alpha=0.9, zorder=2)

        if show_mean_bar:
            for x, cname in ((xb, "occupancy_before"), (xa, "occupancy_after")):
                ax.bar(x, gdf[cname].mean(), width=0.34, color=col, alpha=0.10, zorder=0)

        # group mean ± 95% CI, overlaid just outside each column
        for x, off, cname in ((xb, -0.20, "occupancy_before"),
                              (xa, +0.20, "occupancy_after")):
            m, lo, hi = _mean_ci(gdf[cname].values, conf)
            yerr = [[m - lo], [hi - m]] if np.isfinite(lo) else None
            ax.errorbar(x + off, m, yerr=yerr, fmt="o", color="black", ms=7,
                        capsize=5, lw=1.8, zorder=4)

        dm, dlo, dhi = _mean_ci(gdf["diff"].values, conf)
        ax.text((xb + xa) / 2, 108, g, ha="center", va="bottom",
                fontsize=12, fontweight="bold", color=col)
        ax.text((xb + xa) / 2, 103,
                f"Δ {dm:+.1f}  [{dlo:.1f}, {dhi:.1f}]" if np.isfinite(dlo) else f"Δ {dm:+.1f}",
                ha="center", va="bottom", fontsize=8, color="0.3")
        xticks += [xb, xa]; xlabels += ["before", "after"]

    ax.axhline(33.3, color="grey", ls=":", lw=0.8, alpha=0.5)   # ~one-third reference
    ax.set_xticks(xticks); ax.set_xticklabels(xlabels)
    ax.set_ylabel("social-third occupancy (%)")
    ax.set_ylim(18, 113)
    ax.set_xlim(-0.7, (len(groups) - 1) * cluster_gap + pair_gap + 0.7)
    handles = [Line2D([0],[0], marker="o", color=GROUP_PALETTE[g], lw=0, ms=8, label=g)
               for g in groups]
    handles += [Line2D([0],[0], marker="o", color="black", lw=0, ms=7,
                        label=f"mean ± {int(conf*100)}% CI"),
                Line2D([0],[0], marker="o", color="grey", markerfacecolor="white",
                       lw=0, ms=7, label="low tracking")]
    ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=8)
    ax.set_title("Social-third occupancy: before → after", fontsize=12, pad=30)
    plt.tight_layout()
    plt.show()

plot_occupancy_combined()

In [ ]:
# =============================================================
# CELL 9 — Trajectory traces: one combined figure per group
# =============================================================
# Per group: rows = fish, cols = before/after. Plain trajectory line
# (coloured by PHASE, not by region) over background + arena outline +
# zone dividers. Social third = rightmost after flip. NaN gaps break
# the line. Requires Cells 4, 7 + registry.
# -------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

PHASE_COLOUR = {"before": "#1f77b4", "after": "#d62728"}   # blue / red

def plot_group_traces(group, panel=2.6, lw=0.7, show_dividers=True):
    members = [n for n in loaded if loaded[n]["row"]["group"] == group]
    if not members:
        print(f"No loaded sessions in group {group}"); return
    nrows = len(members)
    fig, axes = plt.subplots(nrows, 2, figsize=(2 * panel * 1.4, panel * nrows),
                             squeeze=False)
    for r, nick in enumerate(members):
        e = loaded[nick]; trj, row = e["trj"], e["row"]
        flip = bool(row.get("flip", False))
        mask = load_roi_mask(row["roi_mask"], flip=flip)
        bg = load_background(row["background"], flip=flip)
        _, (b1, b2), _ = build_zone_map(mask)
        H, W = mask.shape
        phases = {"before": (row["seg_before_start"], row["seg_before_end"]),
                  "after":  (row["seg_after_start"],  row["seg_after_end"])}
        for c, (phase, (s, en)) in enumerate(phases.items()):
            ax = axes[r][c]
            sub = _seg_slice(trj, s, en)
            if bg is not None:
                ax.imshow(bg, cmap="gray", origin="upper")
            ax.contour(mask, levels=[0.5], colors="white", linewidths=0.8)
            if show_dividers:
                ax.axvline(b1, color="cyan", ls="--", lw=0.8)
                ax.axvline(b2, color="cyan", ls="--", lw=0.8)
            ax.plot(sub["x"].values, sub["y"].values,
                    color=PHASE_COLOUR[phase], lw=lw, alpha=0.8)
            ax.set_xlim(0, W); ax.set_ylim(H, 0)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(phase, fontsize=11, fontweight="bold")
            if c == 0:
                ax.set_ylabel(nick, fontsize=6.5, rotation=0,
                              ha="right", va="center", labelpad=6)
    fig.suptitle(f"Trajectories — {group}  (social third = right)",
                 fontsize=13, y=1.0)
    plt.tight_layout()
    plt.show()

for grp in registry.loc[~registry.get("exclude", False), "group"].unique():
    plot_group_traces(grp)

In [ ]:
# =============================================================
# CELL 10 — Pooled group-mean occupancy heatmaps (shared LUT)
# =============================================================
# One before/after pair PER GROUP. Each fish's per-phase occupancy map
# (% of phase time per bin, normalised to its OWN valid frames) is
# averaged across fish -> equal weight per animal. Single LOCKED colour
# scale across ALL panels (CFG["LUT_PERCENTILE"]) for fair comparison.
# Light Gaussian smoothing for a cleaner look (smooth_sigma=0 for raw).
# Requires: loaded (Cell 7); build_zone_map/load_roi_mask/
# load_background/_seg_slice (Cell 4); CFG (Cell 1); scipy.
# -------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

def _occupancy_map(x, y, x_range, y_range, bins):
    """2D histogram normalised to % of that phase's valid frames."""
    valid = ~(np.isnan(x) | np.isnan(y))
    n_valid = int(valid.sum())
    Hm, _, _ = np.histogram2d(x[valid], y[valid], bins=bins, range=[x_range, y_range])
    if n_valid:
        Hm = Hm / n_valid * 100.0
    return Hm, n_valid

def plot_group_mean_maps(groups=None, bins=None, cmap=None, pct_clip=None,
                         smooth_sigma=0.8, show_bg=True):
    bins     = bins     or CFG["HEATMAP_BINS"]
    cmap     = cmap     or CFG["CMAP"]
    pct_clip = pct_clip or CFG["LUT_PERCENTILE"]
    if groups is None:
        groups = list(registry.loc[~registry.get("exclude", False), "group"].unique())
    members = {g: [n for n in loaded if loaded[n]["row"]["group"] == g] for g in groups}
    members = {g: m for g, m in members.items() if m}
    if not members:
        print("No loaded sessions."); return

    first = loaded[next(iter(loaded))]["row"]
    W, H = int(first["w"]), int(first["h"])

    group_maps, group_bg, dividers, allvals = {}, {}, {}, []
    for g, m in members.items():
        for phase in ("before", "after"):
            stack = []
            for nick in m:
                e = loaded[nick]; trj, row = e["trj"], e["row"]
                s, en = row[f"seg_{phase}_start"], row[f"seg_{phase}_end"]
                sub = _seg_slice(trj, s, en)
                Hm, nv = _occupancy_map(sub["x"].values, sub["y"].values, (0, W), (0, H), bins)
                if nv > 0:
                    stack.append(Hm)
            mean_map = np.mean(stack, axis=0) if stack else np.zeros((bins, bins))
            if smooth_sigma:
                mean_map = gaussian_filter(mean_map, sigma=smooth_sigma)
            group_maps[(g, phase)] = mean_map
            allvals.append(mean_map[mean_map > 0])
        bgs = []
        for nick in m:
            row = loaded[nick]["row"]; flip = bool(row.get("flip", False))
            b = load_background(row["background"], flip=flip)
            if b is not None: bgs.append(b)
            _, (b1, b2), _ = build_zone_map(load_roi_mask(row["roi_mask"], flip=flip))
            dividers[g] = (b1, b2)
        group_bg[g] = np.mean(bgs, axis=0) if bgs else None

    flat = np.concatenate(allvals) if allvals else np.array([])
    vmax = max(np.percentile(flat, pct_clip) if flat.size else 1.0, 1e-9)

    n = len(members)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.6 * n), squeeze=False)
    fig.subplots_adjust(top=0.9, hspace=0.28, wspace=0.08, right=0.86)
    im = None
    for r, g in enumerate(members):
        b1, b2 = dividers[g]
        for c, phase in enumerate(("before", "after")):
            ax = axes[r][c]
            if show_bg and group_bg[g] is not None:
                ax.imshow(group_bg[g], cmap="gray", origin="upper", extent=[0, W, H, 0])
            M = group_maps[(g, phase)]
            ax.imshow(np.ma.masked_where(M.T <= 0, M.T), origin="upper",
                      extent=[0, W, H, 0], cmap=cmap, vmin=0, vmax=vmax, alpha=0.9)
            im = ax.images[-1]
            ax.axvline(b1, color="cyan", ls="--", lw=1)
            ax.axvline(b2, color="cyan", ls="--", lw=1)
            ax.set_xlim(0, W); ax.set_ylim(H, 0)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"{g} — {phase}  (n={len(members[g])})",
                         fontsize=11, fontweight="bold")

    cbar = fig.colorbar(im, ax=axes, shrink=0.6, fraction=0.025, pad=0.02)
    cbar.set_label("Mean occupancy (% of phase time per bin) — LOCKED", fontsize=9)
    fig.suptitle("Pooled group-mean occupancy — social third on right, shared LUT",
                 fontsize=14, y=0.97)
    plt.show()
    print(f"Locked scale: vmax = {vmax:.3f}% per bin ({pct_clip}th pct).  "
          + ", ".join(f"{g} (n={len(m)})" for g, m in members.items()))

In [ ]:
plot_group_mean_maps()